# FiBL DOK × PlanetScope — revision analysis (v03, 22 Nov 2026)

This notebook reproduces **every number** of the revised manuscript from the corrected dataset `FIBL_PS_TOAref_2017_2019_v01.csv` (TOA reflectance, daily composites, 4992 records).

It replaces the analysis of the submitted notebook `FIBL_2026 (1).ipynb`, which is kept unchanged as the record of the submitted version.

**Sections**
0. Settings and data loading
1. Data checks (records, duplicates, classes, replicates)
2. Mixed-effects models (replaces Kruskal–Wallis)
3. Classification: 5 classes and two 2-class definitions, replicate × year cross-validation, 20 runs, CI, permutation test
4. Month importance
5. Crop-type baseline
6. Within-plot heterogeneity
7. Sensitivity: date-centred NDVI, other indices, buffer, SVM, grass-clover by cutting period
8. Figures and supplementary tables
9. Save results
8. Final tables (saved as CSV)

Set `QUICK = True` for a fast check (a few minutes); `QUICK = False` reproduces the manuscript values exactly (about 1–2 hours in Colab, mostly the permutation tests).

## Version history
| Version | Date | What changed |
|---|---|---|
| `FIBL_2026 (1).ipynb` | Oct 2026 | Submitted analysis (kept unchanged, GitHub tag `v1.0-submitted`) |
| v01 | 21 Nov 2026 | New analysis with the methods requested by the reviewers |
| v02 | 21 Nov 2026 | Same analysis as v01; adds this history, the change table below and a run log at the end. **No change in any result.** |
| v03 | 22 Nov 2026 | Adds: GEE processing description (section 0b), SVM check (7e), exploratory grass-clover analysis by cutting period (7f), figures and supplementary tables (8). Sections 1-7d unchanged, so all v02 results are reproduced. |

Rule: every change = a new version file; older versions are never overwritten.

## What changed compared with the submitted notebook, and why
| # | Submitted notebook | This notebook | Why (comment) |
|---|---|---|---|
| 1 | Planet radiance used directly as reflectance | TOA reflectance (per-scene `reflectanceCoefficient`), no atmospheric correction | Error found in the audit; R1 1.18 (EVI instability) |
| 2 | 5403 rows, 411 duplicate plot–dates | Daily median composites: 4992 = 96 plots × 52 dates | R1 1.12 |
| 3 | CONMIN coded together with NOFERT ("5-class" matrix had 4 classes) | 5 separate classes: BIODYN, BIOORG, CONFYM, CONMIN, NOFERT | R1 1.21 |
| 4 | Kruskal–Wallis, each date as an independent sample | Linear mixed model per crop: NDVI ~ system + date + replicate, random intercept per plot–year; LR test, BH-FDR, Holm | R1 1.0, 1.16; R2 2 |
| 5 | Cross-validation grouped by plot | Leave one replicate × year out (4 or 8 folds) | R1 1.9 |
| 6 | Windows chosen by maximum accuracy (Table 7) | Windows fixed in advance from phenology; month importance shows where the signal is | R2 1; R1 1.21 |
| 7 | Single run, no uncertainty | 20 seeds, bootstrap 95% CI over groups, permutation test (199) | R2 3 |
| 8 | Only organic vs CONFYM (CONMIN missing) | Three tasks side by side: 5 classes; organic vs CONFYM; organic vs CONFYM + CONMIN | R1 1.21 (no selection after the fact) |
| 9 | Hyperparameters not reported, SVM + RF | RF only: 300 trees, √p features, balanced class weights, not tuned | R1 1.8 |
| 10 | — | Sensitivity: other indices (Table S1), plot buffer −0.5/−1.0/−1.5 m, date-centred NDVI | R1 1.18; R2 4, 6 |
| 11 | Heterogeneity tested with per-date samples | Same mixed-model structure on within-plot SD | R1 1.20 |

The headline 84.4% of the submitted version came from points 1–6 together and is not reproduced.

## 0b. Upstream processing in Google Earth Engine (not run here)
The input CSV was produced in the Google Earth Engine Code Editor with `gee/GEE_fibl_2026_review_v01.js` (main data, 0.5 m buffer) and `GEE_fibl_2026_review_v02.js` (buffers 0.5 / 1.0 / 1.5 m), both in the GitHub repository (`revision_2026-11/gee/`). The PlanetScope scenes and plot polygons are private Earth Engine assets (Planet licence), so this step cannot be run by others; the scripts document it exactly.

| Step | What the script does |
|---|---|
| 1. Scenes | 57 four-band PlanetScope `AnalyticMS` scenes (TOA radiance), Feb 2017 – Sep 2019 |
| 2. Reflectance | Each band × its scene-specific `reflectanceCoefficient` from the Planet metadata (`fibl_reflectance_coefficients_GEE.js`) → TOA reflectance; no atmospheric correction |
| 3. Quality mask | UDM2: keep pixels with clear = 1, cloud = 0, unusable = 0 |
| 4. Indices | NDVI, GNDVI, SAVI (L = 0.5), MSAVI, EVI |
| 5. Daily composite | Median of the scenes of the same day → 52 dates |
| 6. Plot statistics | Each plot shrunk inwards by 0.5 m; mean, standard deviation and count of valid 3 m pixels (weighted by pixel overlap) |
| 7. Filter and export | Keep plot-dates with ≥ 3 valid pixels → 4992 records (96 plots × 52 dates) |

A Python (earthengine-api) port of these steps, checked against the CSV, is planned after resubmission.

In [ ]:
# 0. Settings
QUICK = True            # True: fast check; False: full run used for the manuscript
N_SEEDS = 3 if QUICK else 20      # repeated Random Forest runs per analysis
N_PERM  = 19 if QUICK else 199    # permutations for the significance test
N_BOOT  = 500 if QUICK else 2000  # bootstrap resamples for the 95% CI
N_TREES = 300

# Data file. In Colab: upload the CSV (left panel, Files) or mount Drive and change the path.
DATA = 'FIBL_PS_TOAref_2017_2019_v01.csv'
# Optional buffer-sensitivity files (exports of GEE script fibl_2026_review_v02); leave as they are if not available
BUFFER_FILES = {'-0.5 m': 'FIBL_PS_TOAref_2017_2019_v02.csv',
                '-1.0 m': 'FIBL_PS_TOAref_2017_2019_v02_buf10.csv',
                '-1.5 m': 'FIBL_PS_TOAref_2017_2019_v02_buf15.csv'}

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2, norm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

# Crop windows, fixed in advance from crop phenology (months)
WINDOWS = {'winter wheat 1': [3,4,5,6,7], 'winter wheat 2': [3,4,5,6,7],
           'soybeans': [5,6,7,8,9], 'potatoes': [4,5,6,7,8,9],
           'grass-clover 1': list(range(3,11)), 'grass-clover 2': list(range(3,11))}
# DOK codes -> five analysis classes
CLASS5 = {'D1':'BIODYN','D2':'BIODYN','O1':'BIOORG','O2':'BIOORG',
          'K1':'CONFYM','K2':'CONFYM','M':'CONMIN','N':'NOFERT'}
LABELS5 = ['BIODYN','BIOORG','CONFYM','CONMIN','NOFERT']
# The three classification tasks: which DOK codes enter each
TASKS = {'5 classes': list(CLASS5),
         'organic vs CONFYM': ['D1','D2','O1','O2','K1','K2'],
         'organic vs CONFYM+CONMIN': ['D1','D2','O1','O2','K1','K2','M']}

def load(path):
    d = pd.read_csv(path)
    d['month'] = pd.to_datetime(d['image_date']).dt.month
    d['year'] = d['image_year']
    d['crop'] = d['crop_type']
    d['rep'] = (d['plot_id'] - 1) // 24 + 1          # replicate from plot numbering (verified in section 1)
    d['system'] = d['BIO'].map(CLASS5)
    d['plotyear'] = d['plot_id'].astype(str) + '_' + d['year'].astype(str)
    return d

d = load(DATA)
print(d.shape)

## 1. Data checks
Expected: 4992 records = 96 plots × 52 dates, no duplicate plot–date, five classes, and every replicate × crop strip holds each of the 8 DOK codes exactly once.

In [ ]:
print('records:', len(d), '| plots:', d.plot_id.nunique(), '| dates:', d.image_date.nunique())
print('duplicate plot-date rows:', d.duplicated(['plot_id','image_date']).sum())
print('valid pixels per plot-date (min / median):', d.NDVI_count.min(), '/', d.NDVI_count.median())
print('NDVI range:', round(d.NDVI_mean.min(),3), '-', round(d.NDVI_mean.max(),3))
print(pd.crosstab(d.BIO, d.system_class))
plots = d.drop_duplicates('plot_id')
check = pd.crosstab([plots.rep, plots.GROUP], plots.BIO)
print('replicate check passed:', bool((check.values == 1).all()))
print(plots.groupby('GROUP')[['2017','2018','2019']].first())   # crop strip -> crop per year

## 2. Mixed-effects models
For each crop: `NDVI ~ system + date + replicate`, random intercept per plot-year, fitted by maximum likelihood. The system effect is tested with a likelihood-ratio test (4 df), Benjamini–Hochberg adjusted over the six crops. Pairwise contrasts (each system vs CONFYM, and BIODYN vs BIOORG) are Holm-adjusted within each crop.

In [ ]:
def holm(p):
    p = np.asarray(p); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for k, i in enumerate(order):
        run = max(run, (m - k) * p[i]); adj[i] = min(1, run)
    return adj

def bh(p):
    p = np.asarray(p); order = np.argsort(p)[::-1]; m = len(p); adj = np.empty(m); prev = 1
    for k, i in enumerate(order):
        prev = min(prev, p[i] * m / (m - k)); adj[i] = prev
    return adj

def fit(formula, data):
    for method in ['powell', 'lbfgs', 'nm']:
        try:
            m = smf.mixedlm(formula, data, groups=data['plotyear']).fit(reml=False, method=method)
            if m.converged: return m
        except Exception: pass
    return m

glob, pair = [], []
for crop, months in WINDOWS.items():
    x = d[(d.crop == crop) & d.month.isin(months)].copy(); x['repf'] = x.rep.astype(str)
    m1 = fit("NDVI_mean ~ C(system, Treatment('CONFYM')) + C(image_date) + C(repf)", x)
    m0 = fit("NDVI_mean ~ C(image_date) + C(repf)", x)
    glob.append({'crop': crop, 'LR': 2*(m1.llf - m0.llf), 'p': chi2.sf(2*(m1.llf - m0.llf), 4)})
    names = {k.split('[T.')[1][:-1]: k for k in m1.params.index if 'system' in k}
    cov = m1.cov_params()
    rows = [(s, 'CONFYM', m1.params[names[s]], m1.bse[names[s]]) for s in ['BIODYN','BIOORG','CONMIN','NOFERT']]
    iD, iO = names['BIODYN'], names['BIOORG']
    rows.append(('BIODYN', 'BIOORG', m1.params[iD] - m1.params[iO],
                 np.sqrt(cov.loc[iD,iD] + cov.loc[iO,iO] - 2*cov.loc[iD,iO])))
    praw = [2*norm.sf(abs(e/s)) for _,_,e,s in rows]
    for (a,b,e,s), pr, pa in zip(rows, praw, holm(praw)):
        pair.append({'crop': crop, 'contrast': f'{a} - {b}', 'diff': e, 'p_holm': pa})
lmm_global = pd.DataFrame(glob); lmm_global['p_FDR'] = bh(lmm_global.p)
lmm_pairwise = pd.DataFrame(pair)
print(lmm_global.round(4))
print(lmm_pairwise.pivot(index='crop', columns='contrast', values='diff').round(3))
print(lmm_pairwise.pivot(index='crop', columns='contrast', values='p_holm').round(4))

## 3. Classification
Each plot-year is one sample, described by its monthly mean NDVI within the crop window (gaps filled by interpolation **within the same plot**). Random Forest with fixed hyperparameters. Cross-validation: leave one replicate × year group out (8 folds for two-year crops, 4 for one-year crops). Reported: mean balanced accuracy over `N_SEEDS` runs, approximate 95% CI (bootstrap of whole replicate-year groups), permutation p-value (labels shuffled within replicate-year).

In [ ]:
def features(data, crop, codes, vi='NDVI_mean'):
    x = data[(data.crop == crop) & data.month.isin(WINDOWS[crop]) & data.BIO.isin(codes)]
    F = x.pivot_table(index=['plot_id','year','rep','BIO'], columns='month', values=vi, aggfunc='mean')
    F = F.interpolate(axis=1, limit_direction='both').dropna(axis=1)     # along time, within plot
    return F

def labels(F, task):
    bio = F.index.get_level_values('BIO')
    if task == '5 classes': return np.array([CLASS5[b] for b in bio])
    return np.where(bio.str[0].isin(['D','O']), 'ORGANIC', 'CONVENTIONAL')

def groups(F):
    return np.array([f'{r}_{y}' for r, y in zip(F.index.get_level_values('rep'), F.index.get_level_values('year'))])

def cv_predict(X, y, g, seed, n_trees=N_TREES):
    pred = np.empty(len(y), dtype=object)
    for tr, te in LeaveOneGroupOut().split(X, y, g):
        rf = RandomForestClassifier(n_estimators=n_trees, random_state=seed, class_weight='balanced')
        pred[te] = rf.fit(X[tr], y[tr]).predict(X[te])
    return pred

def evaluate(data, crop, task, vi='NDVI_mean', n_seeds=N_SEEDS, n_perm=N_PERM, n_boot=N_BOOT):
    F = features(data, crop, TASKS[task], vi); X = F.values; y = labels(F, task); g = groups(F)
    preds = [cv_predict(X, y, g, s) for s in range(n_seeds)]
    ba = [balanced_accuracy_score(y, p) for p in preds]
    maj = np.array([pd.Series(col).mode()[0] for col in np.array(preds).T])
    rng = np.random.default_rng(1); G = np.unique(g); boot = []
    for _ in range(n_boot):
        idx = np.concatenate([np.where(g == k)[0] for k in rng.choice(G, len(G))])
        if len(set(y[idx])) == len(set(y)): boot.append(balanced_accuracy_score(y[idx], maj[idx]))
    p = np.nan
    if n_perm:
        hits = 0
        for _ in range(n_perm):
            yp = y.copy()
            for k in G:
                ix = np.where(g == k)[0]; yp[ix] = rng.permutation(yp[ix])
            if balanced_accuracy_score(yp, cv_predict(X, yp, g, 0, 100)) >= np.mean(ba): hits += 1
        p = (hits + 1) / (n_perm + 1)
    return {'crop': crop, 'task': task, 'n': len(y), 'folds': len(G), 'BA': np.mean(ba), 'BA_sd_seeds': np.std(ba),
            'CI_lo': np.percentile(boot, 2.5), 'CI_hi': np.percentile(boot, 97.5), 'perm_p': p}, maj, y

results, confusion = [], np.zeros((5,5), int)
for task in TASKS:
    for crop in WINDOWS:
        r, maj, y = evaluate(d, crop, task); results.append(r)
        if task == '5 classes': confusion += confusion_matrix(y, maj, labels=LABELS5)
        print(task, '|', crop, '| BA', round(r['BA'],3), '| p', r['perm_p'])
classification = pd.DataFrame(results)
print(classification.pivot(index='crop', columns='task', values='BA').round(3))
print(pd.DataFrame(confusion, index=LABELS5, columns=LABELS5))

## 4. Month importance (organic vs CONFYM)
In each held-out fold, the NDVI of one month is randomly permuted; the loss in balanced accuracy shows how much the classification relies on that month.

In [ ]:
imp = []
rng = np.random.default_rng(0)
for crop in WINDOWS:
    F = features(d, crop, TASKS['organic vs CONFYM']); X = F.values; y = labels(F, 'organic vs CONFYM'); g = groups(F)
    drops = np.zeros((N_SEEDS, X.shape[1]))
    for s in range(N_SEEDS):
        pred = np.empty(len(y), dtype=object); perm = [np.empty(len(y), dtype=object) for _ in range(X.shape[1])]
        for tr, te in LeaveOneGroupOut().split(X, y, g):
            rf = RandomForestClassifier(N_TREES, random_state=s, class_weight='balanced').fit(X[tr], y[tr])
            pred[te] = rf.predict(X[te])
            for j in range(X.shape[1]):
                Xp = X[te].copy(); Xp[:, j] = rng.permutation(Xp[:, j]); perm[j][te] = rf.predict(Xp)
        base = balanced_accuracy_score(y, pred)
        drops[s] = [base - balanced_accuracy_score(y, perm[j]) for j in range(X.shape[1])]
    for j, mth in enumerate(F.columns): imp.append({'crop': crop, 'month': int(mth), 'BA_loss': drops[:, j].mean()})
month_importance = pd.DataFrame(imp)
print(month_importance.pivot(index='crop', columns='month', values='BA_loss').round(3))

## 5. Crop-type baseline
Crop type classified from whole-season monthly NDVI (all 12 months), same validation.

In [ ]:
x = d.copy(); x['croptype'] = x.crop.str.replace(r' \d', '', regex=True)
F = x.pivot_table(index=['plot_id','year','rep','croptype'], columns='month', values='NDVI_mean').interpolate(axis=1, limit_direction='both').dropna(axis=1)
y = F.index.get_level_values('croptype').values; g = groups(F)
preds = [cv_predict(F.values, y, g, s) for s in range(N_SEEDS)]
print('crop type BA:', round(np.mean([balanced_accuracy_score(y, p) for p in preds]), 3), '| plot-years:', len(y))
L = sorted(set(y)); print(pd.DataFrame(confusion_matrix(y, preds[0], labels=L), index=L, columns=L))

## 6. Within-plot heterogeneity
Within-plot NDVI standard deviation compared among systems with the same mixed-model structure.

In [ ]:
het = []
for crop, months in WINDOWS.items():
    x = d[(d.crop == crop) & d.month.isin(months)].copy(); x['repf'] = x.rep.astype(str)
    m1 = fit("NDVI_stdDev ~ C(system, Treatment('CONFYM')) + C(image_date) + C(repf)", x)
    m0 = fit("NDVI_stdDev ~ C(image_date) + C(repf)", x)
    het.append({'crop': crop, 'p_system': chi2.sf(2*(m1.llf - m0.llf), 4), **x.groupby('system').NDVI_stdDev.mean().round(4).to_dict()})
heterogeneity = pd.DataFrame(het); print(heterogeneity)

## 7. Sensitivity analyses
(a) **Date-centred NDVI**: subtracting the crop mean on each date removes everything common to all plots on that date (atmosphere, sun angle, sensor). (b) **Other indices** instead of NDVI. (c) **Buffer**: only if the three GEE v02 exports are present.

In [ ]:
# (a) date-centred NDVI
d['NDVI_centred'] = d.NDVI_mean - d.groupby(['crop','image_date']).NDVI_mean.transform('mean')
centred = [{'crop': c, 'BA_centred': evaluate(d, c, 'organic vs CONFYM', 'NDVI_centred', n_perm=0, n_boot=10)[0]['BA']}
           for c in ['winter wheat 1','winter wheat 2','soybeans','potatoes']]
print(pd.DataFrame(centred).round(3))

# (b) other indices
other = []
for vi in ['GNDVI_mean','SAVI_mean','MSAVI_mean','EVI_mean']:
    for task in ['5 classes','organic vs CONFYM']:
        for crop in WINDOWS:
            other.append({'index': vi, 'task': task, 'crop': crop,
                          'BA': evaluate(d, crop, task, vi, n_perm=0, n_boot=10)[0]['BA']})
other_indices = pd.DataFrame(other)
print(other_indices.pivot_table(index=['task','crop'], columns='index', values='BA').round(3))

# (c) buffer
if all(os.path.exists(f) for f in BUFFER_FILES.values()):
    rows = []
    for name, f in BUFFER_FILES.items():
        db = load(f)
        for crop in ['winter wheat 1','winter wheat 2','soybeans','potatoes']:
            rows.append({'buffer': name, 'crop': crop, 'BA': evaluate(db, crop, 'organic vs CONFYM', n_perm=0, n_boot=10)[0]['BA']})
    buffer_ba = pd.DataFrame(rows); print(buffer_ba.pivot(index='crop', columns='buffer', values='BA').round(3))
else:
    buffer_ba = pd.DataFrame(); print('Buffer files not found: buffer sensitivity skipped.')

### 7e. Classifier check: support vector machine
Same features, windows, tasks and cross-validation as the Random Forest. Settings fixed in advance, not tuned: RBF kernel, C = 1, gamma = scikit-learn default, standardised features, balanced class weights. The SVM is deterministic, so one run; group-bootstrap CI and permutation test as for the Random Forest.

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def cv_svm(X, y, g):
    pred = np.empty(len(y), dtype=object)
    for tr, te in LeaveOneGroupOut().split(X, y, g):
        m = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced'))
        pred[te] = m.fit(X[tr], y[tr]).predict(X[te])
    return pred

rows = []; rng = np.random.default_rng(1)
for task in TASKS:
    for crop in WINDOWS:
        F = features(d, crop, TASKS[task]); X = F.values; y = labels(F, task); g = groups(F); G = np.unique(g)
        p = cv_svm(X, y, g); ba = balanced_accuracy_score(y, p)
        boot = []
        for _ in range(N_BOOT):
            idx = np.concatenate([np.where(g == k)[0] for k in rng.choice(G, len(G))])
            if len(set(y[idx])) == len(set(y)): boot.append(balanced_accuracy_score(y[idx], p[idx]))
        hits = 0
        for _ in range(N_PERM):
            yp = y.copy()
            for k in G:
                ix = np.where(g == k)[0]; yp[ix] = rng.permutation(yp[ix])
            if balanced_accuracy_score(yp, cv_svm(X, yp, g)) >= ba: hits += 1
        rows.append({'task': task, 'crop': crop, 'SVM_BA': ba, 'CI_lo': np.percentile(boot, 2.5),
                     'CI_hi': np.percentile(boot, 97.5), 'perm_p': (hits + 1) / (N_PERM + 1)})
svm_results = pd.DataFrame(rows)
cmp = svm_results.merge(classification[['task','crop','BA']].rename(columns={'BA':'RF_BA'}), on=['task','crop'])
print(cmp[['task','crop','RF_BA','SVM_BA','perm_p']].round(3).to_string(index=False))

### 7f. Grass-clover by cutting period (exploratory)
Defined after the main results were known, so it is reported as exploratory. Apparent cuts = dates (from April on) where the mean NDVI of all grass-clover plots drops by ≥ 0.08. From the start of the season to the 1st and to the 2nd cut: mixed model (random intercept per plot) and Random Forest organic vs CONFYM on the per-date NDVI, leave-one-replicate-out (4 folds), with a permutation test. Four windows are tested in total, so a correction for 4 tests applies.

In [ ]:
N_PERM_GC = 19 if QUICK else 99
gc_rows = []
for crop in ['grass-clover 1', 'grass-clover 2']:
    g0 = d[d.crop == crop].copy(); g0['date'] = pd.to_datetime(g0.image_date)
    m = g0.groupby('date').NDVI_mean.mean().sort_index()
    cuts = [t for t, v in m.diff().items() if v <= -0.08 and t.month >= 4]
    print(crop, 'apparent cuts (first date after the drop):', [str(c.date()) for c in cuts])
    for w, cut in [('start to 1st cut', cuts[0]), ('start to 2nd cut', cuts[1])]:
        s = g0[g0.date < cut].copy(); s['repf'] = s.rep.astype(str)
        f1 = smf.mixedlm("NDVI_mean ~ C(system, Treatment('CONFYM')) + C(image_date) + C(repf)", s, groups=s.plot_id).fit(reml=False, method='powell')
        f0 = smf.mixedlm("NDVI_mean ~ C(image_date) + C(repf)", s, groups=s.plot_id).fit(reml=False, method='powell')
        co = {k.split('T.')[1].rstrip(']'): (f1.params[k], f1.pvalues[k]) for k in f1.params.index if 'system' in k}
        c = s[s.system.isin(['BIODYN', 'BIOORG', 'CONFYM'])]
        X = c.pivot_table(index='plot_id', columns='image_date', values='NDVI_mean')
        meta = c.groupby('plot_id').agg(system=('system', 'first'), rep=('rep', 'first')).loc[X.index]
        y = (meta.system == 'CONFYM').astype(int).values; grp = meta.rep.values; Xv = X.values
        def cvg(yy, seed):
            pred = np.zeros_like(yy)
            for r in np.unique(grp):
                tr = grp != r
                pred[~tr] = RandomForestClassifier(N_TREES, class_weight='balanced', random_state=seed).fit(Xv[tr], yy[tr]).predict(Xv[~tr])
            return balanced_accuracy_score(yy, pred)
        ba = np.mean([cvg(y, s_) for s_ in range(N_SEEDS)])
        rng = np.random.default_rng(0); hits = 0
        for k in range(N_PERM_GC):
            yp = y.copy()
            for r in np.unique(grp):
                ix = np.where(grp == r)[0]; yp[ix] = rng.permutation(yp[ix])
            if cvg(yp, k) >= ba: hits += 1
        gc_rows.append({'crop': crop, 'window': w, 'to': str(s.date.max().date()), 'n_dates': s.date.nunique(),
                        'p_system': chi2.sf(2*(f1.llf - f0.llf), 4), 'BIODYN-CONFYM': co['BIODYN'][0], 'BIOORG-CONFYM': co['BIOORG'][0],
                        'NOFERT-CONFYM': co['NOFERT'][0], 'BA_org_vs_CONFYM': ba, 'perm_p': (hits + 1) / (N_PERM_GC + 1)})
gc_cuts = pd.DataFrame(gc_rows); print(gc_cuts.round(3).to_string(index=False))

## 8. Figures and supplementary tables
Figure files are written to the results folder of this run (section 9).

In [ ]:
import matplotlib.pyplot as plt
tag = pd.Timestamp.now().strftime('%Y-%m-%d_%H%M') + ('_QUICK' if QUICK else '_FULL')
out = f'results_{tag}'; os.makedirs(out, exist_ok=True)
COL = {'BIODYN':'#7B3294','BIOORG':'#1B7837','CONFYM':'#C2185B','CONMIN':'#E08214','NOFERT':'#5A5A5A'}
plt.rcParams.update({'font.size': 9})
d['date'] = pd.to_datetime(d.image_date)
# Figure: NDVI profiles with 95% CI across plots, crop window shaded
fig, axs = plt.subplots(3, 2, figsize=(10, 10.5))
for ax, crop in zip(axs.flat, WINDOWS):
    x = d[d.crop == crop]
    for yr, xy in x.groupby('year'):
        for s in LABELS5:
            gg = xy[xy.system == s].groupby('date').NDVI_mean; mm = gg.mean(); se = gg.std() / np.sqrt(gg.count()); doy = mm.index.dayofyear
            ax.plot(doy, mm.values, '-o', ms=2.5, lw=1.2, color=COL[s], label=s if yr == x.year.min() else None)
            ax.fill_between(doy, (mm - 1.96*se).values, (mm + 1.96*se).values, color=COL[s], alpha=0.15, lw=0)
    lo, hi = WINDOWS[crop][0], WINDOWS[crop][-1]
    ax.axvspan(pd.Timestamp(2018, lo, 1).dayofyear, (pd.Timestamp(2018, hi, 1) + pd.offsets.MonthEnd(0)).dayofyear, color='#DDDDDD', alpha=0.5, zorder=0)
    ax.set_title(f"{crop.capitalize()} ({', '.join(str(v) for v in sorted(x.year.unique()))})"); ax.set_ylabel('NDVI (TOA reflectance)')
    ax.set_xticks([pd.Timestamp(2018, m_, 1).dayofyear for m_ in range(1, 13)]); ax.set_xticklabels(list('JFMAMJJASOND')); ax.set_xlim(30, 335); ax.grid(alpha=0.25)
axs[0, 0].legend(frameon=False, fontsize=8, loc='lower left'); fig.tight_layout(); fig.savefig(f'{out}/fig_profiles.png', dpi=300); plt.show()
# Figure: month importance
mn = ['','Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
fig, axs = plt.subplots(2, 3, figsize=(11, 6), sharey=True)
for ax, crop in zip(axs.flat, WINDOWS):
    t = month_importance[month_importance.crop == crop].sort_values('month')
    ax.bar([mn[m_] for m_ in t.month], t.BA_loss, color=['#C2185B' if v == t.BA_loss.max() else '#9E9E9E' for v in t.BA_loss])
    ax.axhline(0, color='k', lw=0.6); ax.set_title(crop.capitalize()); ax.grid(axis='y', alpha=0.25)
for ax in axs[:, 0]: ax.set_ylabel('Loss in balanced accuracy')
fig.tight_layout(); fig.savefig(f'{out}/fig_month_importance.png', dpi=300); plt.show()
# Figure: pooled five-class confusion matrix
cm = pd.DataFrame(confusion, index=LABELS5, columns=LABELS5); r = cm.div(cm.sum(1), axis=0)
fig, ax = plt.subplots(figsize=(5.2, 4.4)); im = ax.imshow(r.values, cmap='Blues', vmin=0, vmax=1)
for i in range(5):
    for j in range(5): ax.text(j, i, int(cm.values[i, j]), ha='center', va='center', color='white' if r.values[i, j] > 0.5 else 'black')
ax.set_xticks(range(5)); ax.set_xticklabels(LABELS5, rotation=30, ha='right'); ax.set_yticks(range(5)); ax.set_yticklabels(LABELS5)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); fig.colorbar(im, ax=ax, label='Share of true class', fraction=0.046)
fig.tight_layout(); fig.savefig(f'{out}/fig_confusion_5class.png', dpi=300); plt.show()
# Supplementary tables
s1 = pd.concat([other_indices, classification[classification.task.isin(['5 classes','organic vs CONFYM'])].assign(index='NDVI_mean')[['index','task','crop','BA']]])
tableS1 = s1.pivot_table(index=['task','crop'], columns='index', values='BA')[['NDVI_mean','GNDVI_mean','SAVI_mean','MSAVI_mean','EVI_mean']].round(2)
tableS2 = cmp[['task','crop','RF_BA','SVM_BA','perm_p']].round(3)
print('Table S1'); print(tableS1); print('Table S2'); print(tableS2.to_string(index=False))

## 9. Save the final tables
Each run writes new files with the date and mode in the name; nothing is overwritten.

In [ ]:
for name, df in {'lmm_global': lmm_global, 'lmm_pairwise': lmm_pairwise, 'classification': classification,
                 'month_importance': month_importance, 'heterogeneity': heterogeneity, 'other_indices': other_indices,
                 'centred': pd.DataFrame(centred), 'buffer': buffer_ba, 'svm_vs_rf': cmp, 'grassclover_cuts': gc_cuts,
                 'tableS1_indices': tableS1.reset_index(), 'tableS2_svm': tableS2}.items():
    df.to_csv(f'{out}/{name}.csv', index=False)
pd.DataFrame(confusion, index=LABELS5, columns=LABELS5).to_csv(f'{out}/confusion_5class_pooled.csv')
print('saved to', out, sorted(os.listdir(out)))

## Run log

In [ ]:
# Run log: paste this output into the CHANGELOG when you run the full version
import datetime, platform, sklearn, statsmodels
print('run at        :', datetime.datetime.now().strftime('%Y-%m-%d %H:%M'))
print('notebook      : FIBL_revision_analysis_v03_2026-11-22.ipynb')
print('mode          :', 'QUICK' if QUICK else 'FULL', '| seeds', N_SEEDS, '| perms', N_PERM, '| boot', N_BOOT)
print('data          :', DATA)
print('python        :', platform.python_version(), '| sklearn', sklearn.__version__, '| statsmodels', statsmodels.__version__)